# Cleaning ACS Data

**Goal:**  
Prepare the ACS data so it can be used later in the housing affordability analysis.

**Plan:**
1. Load the raw ACS files
2. Combine the yearly files into one dataset
3. Check the combined data
4. Rename and organize the columns
5. Fix data types
6. Clean county and FIPS columns
7. Create useful housing variables
8. Convert ACS age groups into approximate generations
9. Finalize ACS generation data using CBSA codes
10. Finalize ACS county-year data

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

### Step 1: Loading Raw ACS Files

This step loads the yearly ACS files from the ACS data folder. These files only include New Jersey counties.

In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

acs_path = project_root / "data" / "ACS"
clean_path = project_root / "clean_data"

clean_path.mkdir(exist_ok=True)

acs_files = sorted([
    file for file in acs_path.glob("acs*.csv")
    if file.name[3:7].isdigit() and "-" not in file.stem
])

if len(acs_files) == 0:
    raise FileNotFoundError("No ACS files found. Make sure files like acs2005.csv are in data/ACS.")

print("CSV files found:")
for file in acs_files:
    print(file.name)

CSV files found:
acs2005-NJ-PA-DE-NY-MD.csv
acs2006-NJ-PA-DE-NY-MD.csv
acs2007-NJ-PA-DE-NY-MD.csv
acs2008-NJ-PA-DE-NY-MD.csv
acs2009-NJ-PA-DE-NY-MD.csv
acs2010-NJ-PA-DE-NY-MD.csv
acs2011-NJ-PA-DE-NY-MD.csv
acs2012-NJ-PA-DE-NY-MD.csv
acs2013-NJ-PA-DE-NY-MD.csv
acs2014-NJ-PA-DE-NY-MD.csv
acs2015-NJ-PA-DE-NY-MD.csv
acs2016-NJ-PA-DE-NY-MD.csv
acs2017-NJ-PA-DE-NY-MD.csv
acs2018-NJ-PA-DE-NY-MD.csv
acs2019-NJ-PA-DE-NY-MD.csv
acs2021-NJ-PA-DE-NY-MD.csv
acs2022-NJ-PA-DE-NY-MD.csv
acs2023-NJ-PA-DE-NY-MD.csv


### Step 2: Combining the Files

This step combines the yearly ACS files into one dataframe and adds a `year` column from each file name.

In [ ]:
dfs = []

for file in acs_files:
    df = pd.read_csv(file, dtype=str)
    year = int(file.name[3:7])
    df["year"] = year
    dfs.append(df)

acs = pd.concat(dfs, ignore_index=True)

acs.head()

ValueError: invalid literal for int() with base 10: '2005-NJ-PA-DE-NY-MD'

In [ ]:
acs.shape

This shows that the NJ ACS files were loaded correctly. Each row is one county in one year.

### Step 3: Checking the Combined Data

In [ ]:
# Check data types and non-null counts
acs.info()

In [ ]:
# Check column names
acs.columns

In [ ]:
# Check missing values
acs.isnull().sum()

### Step 4: Renaming Columns

The ACS columns use Census codes, so I renamed them to make the data easier to read and use.

In [ ]:
rename_cols = {
    "NAME": "county_name",
    "B25007_001E": "total_occupied_units",
    "B25007_002E": "owner_occupied_total",
    "B25007_003E": "owner_occupied_hh_15_24",
    "B25007_004E": "owner_occupied_hh_25_34",
    "B25007_005E": "owner_occupied_hh_35_44",
    "B25007_006E": "owner_occupied_hh_45_54",
    "B25007_007E": "owner_occupied_hh_55_59",
    "B25007_008E": "owner_occupied_hh_60_64",
    "B25007_009E": "owner_occupied_hh_65_74",
    "B25007_010E": "owner_occupied_hh_75_84",
    "B25007_011E": "owner_occupied_hh_85_plus",
    "B25007_012E": "renter_occupied_total",
    "B25007_013E": "renter_occupied_hh_15_24",
    "B25007_014E": "renter_occupied_hh_25_34",
    "B25007_015E": "renter_occupied_hh_35_44",
    "B25007_016E": "renter_occupied_hh_45_54",
    "B25007_017E": "renter_occupied_hh_55_59",
    "B25007_018E": "renter_occupied_hh_60_64",
    "B25007_019E": "renter_occupied_hh_65_74",
    "B25007_020E": "renter_occupied_hh_75_84",
    "B25007_021E": "renter_occupied_hh_85_plus",
    "state": "state_fips",
    "county": "county_fips"
}

acs = acs.rename(columns=rename_cols)

acs.head()

Each row is one county-year. Owner and renter columns count occupied housing units by the age of the householder. `hh` means householder.

### Step 5: Fixing Data Types

In [ ]:
acs.dtypes

The count columns need to be numeric so they can be used for calculations later.

In [ ]:
count_cols = [
    col for col in acs.columns
    if col.startswith("total_occupied_units")
    or col.startswith("owner_occupied")
    or col.startswith("renter_occupied")
]

for col in count_cols:
    acs[col] = pd.to_numeric(acs[col], errors="coerce")

acs["year"] = pd.to_numeric(acs["year"], errors="coerce").astype(int)
acs["state_fips"] = acs["state_fips"].astype(str).str.zfill(2)
acs["county_fips"] = acs["county_fips"].astype(str).str.zfill(3)

acs.dtypes


### Step 6: Cleaning County and FIPS Columns

This step cleans the county name and creates a full county FIPS code so the ACS data can merge with the other datasets later.

In [ ]:
acs["county_clean"] = (
    acs["county_name"]
    .str.replace(" County, New Jersey", "", regex=False)
    .str.strip()
)

acs["county_fips_full"] = acs["state_fips"].str.zfill(2) + acs["county_fips"].str.zfill(3)

acs[["county_name", "county_clean", "state_fips", "county_fips", "county_fips_full", "year"]].head()

### Step 7: Creating Useful ACS Variables

This step creates simple rates that are easier to compare across counties and years.

In [ ]:
acs["owner_rate"] = acs["owner_occupied_total"] / acs["total_occupied_units"]
acs["renter_rate"] = acs["renter_occupied_total"] / acs["total_occupied_units"]

acs[[
    "county_clean",
    "year",
    "total_occupied_units",
    "owner_occupied_total",
    "renter_occupied_total",
    "owner_rate",
    "renter_rate"
]].head()

### Step 8: Converting Age Groups Into Generations


In [ ]:
generation_ranges = [
    ("Greatest/Older", 0, 1927),
    ("Silent", 1928, 1945),
    ("Baby Boomer", 1946, 1964),
    ("Gen X", 1965, 1980),
    ("Millennial", 1981, 1996),
    ("Gen Z", 1997, 2012),
]

possible_age_group_cols = {
    "owner_occupied_hh_15_24": ("owner", 15, 24),
    "owner_occupied_hh_25_34": ("owner", 25, 34),
    "owner_occupied_hh_35_44": ("owner", 35, 44),
    "owner_occupied_hh_45_54": ("owner", 45, 54),
    "owner_occupied_hh_55_59": ("owner", 55, 59),
    "owner_occupied_hh_60_64": ("owner", 60, 64),
    "owner_occupied_hh_65_74": ("owner", 65, 74),
    "owner_occupied_hh_75_84": ("owner", 75, 84),
    "owner_occupied_hh_85_plus": ("owner", 85, 100),

    "renter_occupied_hh_15_24": ("renter", 15, 24),
    "renter_occupied_hh_25_34": ("renter", 25, 34),
    "renter_occupied_hh_35_44": ("renter", 35, 44),
    "renter_occupied_hh_45_54": ("renter", 45, 54),
    "renter_occupied_hh_55_59": ("renter", 55, 59),
    "renter_occupied_hh_60_64": ("renter", 60, 64),
    "renter_occupied_hh_65_74": ("renter", 65, 74),
    "renter_occupied_hh_75_84": ("renter", 75, 84),
    "renter_occupied_hh_85_plus": ("renter", 85, 100),
}

age_group_cols = {
    col: values
    for col, values in possible_age_group_cols.items()
    if col in acs.columns
}

def get_generation(birth_year):
    for generation, start_year, end_year in generation_ranges:
        if start_year <= birth_year <= end_year:
            return generation
    return "Unknown"

def generation_weights_for_age_group(year, min_age, max_age):
    ages = range(min_age, max_age + 1)
    generation_counts = {}

    for age in ages:
        birth_year = year - age
        generation = get_generation(birth_year)
        generation_counts[generation] = generation_counts.get(generation, 0) + 1

    total_ages = max_age - min_age + 1

    return {
        generation: count / total_ages
        for generation, count in generation_counts.items()
    }

id_cols = [
    "county_clean",
    "county_fips_full",
    "year",
]

generation_rows = []

for _, row in acs.iterrows():
    year = int(row["year"])
    base_values = {col: row[col] for col in id_cols}

    for source_col, (tenure, min_age, max_age) in age_group_cols.items():
        source_count = row[source_col]
        weights = generation_weights_for_age_group(year, min_age, max_age)

        for generation, weight in weights.items():
            new_row = base_values.copy()
            new_row.update({
                "generation": generation,
                "tenure": tenure,
                "housing_units_estimated": source_count * weight,
            })
            generation_rows.append(new_row)

acs_generation_long = pd.DataFrame(generation_rows)

acs_generation_long = (
    acs_generation_long
    .groupby(id_cols + ["generation", "tenure"], as_index=False)["housing_units_estimated"]
    .sum()
)

acs_generation = (
    acs_generation_long
    .pivot_table(
        index=id_cols + ["generation"],
        columns="tenure",
        values="housing_units_estimated",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

acs_generation.columns.name = None

acs_generation = acs_generation.rename(columns={
    "owner": "owner_units_estimated",
    "renter": "renter_units_estimated",
})

if "owner_units_estimated" not in acs_generation.columns:
    acs_generation["owner_units_estimated"] = 0

if "renter_units_estimated" not in acs_generation.columns:
    acs_generation["renter_units_estimated"] = 0

acs_generation["total_generation_units"] = (
    acs_generation["owner_units_estimated"] +
    acs_generation["renter_units_estimated"]
)

acs_generation = acs_generation[
    [
        "county_clean",
        "county_fips_full",
        "year",
        "generation",
        "owner_units_estimated",
        "renter_units_estimated",
        "total_generation_units",
    ]
]

acs_generation = acs_generation.sort_values(
    ["year", "county_clean", "generation"]
).reset_index(drop=True)

duplicate_check = acs_generation.duplicated(
    subset=["county_fips_full", "year", "generation"]
).sum()

print("Shape:", acs_generation.shape)
print("Duplicate county-year-generation rows:", duplicate_check)

acs_generation.head()

### Step 9: Finalizing ACS Generation Data

This step uses CBSA codes for the generation file. The data still only includes New Jersey counties, so CBSA totals represent the NJ portion of each CBSA.

In [ ]:
possible_cbsa_paths = [
    project_root / "data" / "county_to_cbsa.csv",
    project_root / "data" / "CBSA" / "county_to_cbsa.csv",
]

cbsa_path = next((path for path in possible_cbsa_paths if path.exists()), None)

if cbsa_path is None:
    raise FileNotFoundError("county_to_cbsa.csv was not found in data/ or data/CBSA/.")

county_cbsa = pd.read_csv(cbsa_path, dtype=str)
county_cbsa.columns = county_cbsa.columns.str.strip()

county_cbsa = county_cbsa.rename(columns={
    "County FIPS": "county_fips",
    "county_fips_full": "county_fips",
    "FIPS": "county_fips",
    "CBSA Code": "cbsa_code",
    "CBSA": "cbsa_code",
    "CBSA Title": "cbsa_name",
    "CBSA Name": "cbsa_name"
})

county_cbsa = county_cbsa[["county_fips", "cbsa_code", "cbsa_name"]].drop_duplicates()
county_cbsa["county_fips"] = county_cbsa["county_fips"].astype(str).str.zfill(5)
county_cbsa["cbsa_code"] = county_cbsa["cbsa_code"].astype(str)

acs_generation["county_fips_full"] = acs_generation["county_fips_full"].astype(str).str.zfill(5)

acs_generation_cbsa_work = acs_generation.merge(
    county_cbsa,
    left_on="county_fips_full",
    right_on="county_fips",
    how="left"
)

missing_cbsa = acs_generation_cbsa_work["cbsa_code"].isnull().sum()
print("Rows missing CBSA:", missing_cbsa)

acs_generation_cbsa = (
    acs_generation_cbsa_work
    .dropna(subset=["cbsa_code"])
    .groupby(["cbsa_code", "cbsa_name", "year", "generation"], as_index=False)
    .agg({
        "owner_units_estimated": "sum",
        "renter_units_estimated": "sum",
        "total_generation_units": "sum"
    })
)

cbsa_year_totals = (
    acs_generation_cbsa
    .groupby(["cbsa_code", "year"])["total_generation_units"]
    .transform("sum")
)

acs_generation_cbsa["generation_share_of_total_units"] = np.where(
    cbsa_year_totals > 0,
    acs_generation_cbsa["total_generation_units"] / cbsa_year_totals,
    np.nan
)

acs_generation_cbsa["owner_rate_within_generation"] = np.where(
    acs_generation_cbsa["total_generation_units"] > 0,
    acs_generation_cbsa["owner_units_estimated"] / acs_generation_cbsa["total_generation_units"],
    np.nan
)

acs_generation_cbsa["renter_rate_within_generation"] = np.where(
    acs_generation_cbsa["total_generation_units"] > 0,
    acs_generation_cbsa["renter_units_estimated"] / acs_generation_cbsa["total_generation_units"],
    np.nan
)

acs_generation_cbsa = acs_generation_cbsa[
    [
        "cbsa_code",
        "cbsa_name",
        "year",
        "generation",
        "owner_units_estimated",
        "renter_units_estimated",
        "total_generation_units",
        "generation_share_of_total_units",
        "owner_rate_within_generation",
        "renter_rate_within_generation",
    ]
]

acs_generation_cbsa = acs_generation_cbsa.sort_values(
    ["year", "cbsa_name", "generation"]
).reset_index(drop=True)

acs_generation_cbsa.to_csv(clean_path / "ACS_generation_CBSA_NJ_only_clean.csv", index=False)

print("Shape:", acs_generation_cbsa.shape)

acs_generation_cbsa.head()

In [ ]:
acs_generation_cbsa.shape

### Step 10: Finalizing ACS

This creates the county-year ACS file for the main merge. It keeps only the 21 New Jersey counties.

In [ ]:
acs_county_year = acs[acs["state_fips"].astype(str).str.zfill(2) == "34"].copy()

acs_county_year = acs_county_year[
    [
        "county_fips_full",
        "county_clean",
        "year",
        "total_occupied_units",
        "owner_occupied_total",
        "renter_occupied_total",
        "owner_rate",
        "renter_rate",
    ]
]

acs_county_year = acs_county_year.rename(columns={
    "county_fips_full": "county_fips",
    "county_clean": "county"
})

acs_county_year["county_fips"] = acs_county_year["county_fips"].astype(str).str.zfill(5)

acs_county_year = acs_county_year.sort_values(
    ["year", "county"]
).reset_index(drop=True)

acs_county_year.to_csv(clean_path / "ACS_county_year_clean.csv", index=False)

print("Shape:", acs_county_year.shape)
print("Counties:", acs_county_year["county"].nunique())
print("Duplicates:", acs_county_year.duplicated(subset=["county_fips", "year"]).sum())

acs_county_year.head()